# Decision Appendix: `02_profile_analysis.ipynb`

**Author:** Amy Zhang  
**AI Collaborators:** ChatGPT, Claude Code, Perplexity AI  
**Date:** January - February 2026

---

# 1. **Import Libraries & Dataset**  
---
# 2. **Data Shape: Finding the Observation Grain**  

#### **Notes from Coverage Heatmap & Summary: Thinking Through Column Names** 

- Coverage for metric/measurement-type columns range from highly irregular to effectively absent. This would seem to make the dataset impossible to use for analysis, but the fact that EIA continues to compile and provide these year-to-year encourages us to suspend final judgement.
- `_consumption` columns (a little before being half-way down the y-axis) are prefixed by different energy resources: the irregularity of data suggests that not each one applies to each 'thing' that is being observed per row. If this column were to be 'melted' (ie one 'consumption' column, with prefixes now values), to what would they pertain?
- Right before `_consumption` columns, another hint: a group of `fuel_consumption_from_` columns. The `fuel_consumption_from_all_fuel_types` would seem to be the summary column for the rest (which presumably provide the breakdown by equipment technology: steam turbines, single shaft combined cycle units, combined cycle gas turbines -- all plural), hence containing more data than the rest. This suggests the dimension is at the level of something that can involve a combination of these different technologies. Domain research required.
- The hypothesis that the grain of the dataset is at the level of something that can involve a combination of turbines, single shaft combined cycle units, and combined cycle gas turbines simultaneouosly is further reinforced by columns that expand on these dimensions-- for each, detailing `summer_capacity_`, `gross_generation_`, `net_generation_`
- While the columns mentioned thus far are all `float` dtype, some -- particularly `summer_capacity_`, `gross_generation_`, `net_generation_` -- might be descriptive or attribute-stable rather than dynamic measurements being tracked. In other words, in an ERD, they would be part of a dimension table rather than a fact table.
- `fuel_consumption` would seem to be a fact table metric; in this dataset on Thermoelectric Cooling, it would seem to be narrated in relation to water use metrics, the more complete ones being `water_withdrawal_volume_` and `water_consumption_volume`. Domain research required to understand the distinction. If they relate to a particular technology (similar to steam turbines v. single or combined shafts, etc), might these be 'unmelted'; ie might irregular reporting actually speak of flattened dimensionality?
- At this point, we realize that, since this is Thermoelectric Cooling data, the unit of analysis is an individual cooling system. There are columns that would seem to be specific to a dimension table for this cooling system: `cooling_type_`, `cooling_system_type`, etc. However, there is no given unique 'cooling_system_id' column: the fact that we are given `generator_id`, `boiler_id`, and `cooling_id`, alongside the fact that the closest indication of a unique cooling system piece at work is `cooling_unit_hours_in_service` ('unit' in the singular form) suggests this table to be more granular than the aggregate cooling system of a plant, but rather each of its units. Domain research required. 

#### **Domain Questions (ordered by priority)** 
1) What is the difference between water withdrawal and water consumption?
2) What are the types of cooling systems (`cooling_system_type`)? How do they relate to some combination of steam turbines and shaft-based technologies?
3) Does `cooling_unit_hours_in_service` relate specifically to `cooling_id`? Is it that a generator and a boiler are necessary for the operation of a cooling unit, or is it that we need to track which generator and/or boiler is being cooled by a cooling unit?
4) What do the values of `generator_primary_technology` mean? 

#### **Data Questions**
1) Confirm simultaneity for generation equipments-- do rows tend to have data for more than one technology at a time (something in a steam turbine column, as well as a shaft-based one)?
2) Does each unique `cooling_id` for a unique `plant_id` have one or more `generator_id` and/or `boiler_id`
3) Does a time-series analysis of `cooling_unit_hours_in_service` for a unique `cooling_id` for a unique `plant_id` plot coherently?
4) How many dimensions is this series?


_Current hypothesis: We are looking at the story of what each piece of energy producing unit in a plant requires cooling to stay thermodynamically efficient. This is why the dataset columns unfold (from left to right) as energy generation consumption to water usage needs. Perhaps even more importantly, there might be multiple potential granularities, with columns belonging to different grains (hence data gaps)-- there might not be a single observation row 'truth'; cleaning and analysis will require a grain selection beforehand._


##   ***2a) Investigating Multiple Technology-Type Columns***  

> ```
> Rows by technology count:
> 0      3930
> 1    913506
> Name: count, dtype: int64
> 
> Rows with multiple technologies: 0
>```

#### **Notes**
- The technology-related columns appear to be mutually exclusive; each row’s subject seems to correspond to a single technology type at a time.
- Open questions / next checks:
  1) Is the technology type uniquely determined by each `generator_id`?
  2) If not, the columns toward the end of the table suggest that the effective entity may be defined by a combination of generator, boiler, and cooling unit (`generator_*`, `boiler_*`, `cooling_*`). In that case, a composite unique key will be tested, assuming technology type is not a fixed attribute of a unique `generator_id`.
  3) 3,930 rows without technology type: pending further exploration / possible cleaning

> ```
> Technology types per plant_code - generator_id:
> technology_type
> 0       5
> 1    3397
> Name: count, dtype: int64
> 
> Plant Generators with multiple tech types: 0
> ```

#### **Notes**
- No plant generators have multiple tech types; this confirms that `technology_type` (i.e., its many sub-columns) is an attribute of `plant_id`-`generator_id`.
- The dataset contains ~917k rows but only 3,402 unique `plant_id`-`generator_id`.
- Therefore, the grain of the dataset is finer than the generator level.

#### **Questions**
- If we take `plant_id`-`generator_id` as the grain, then the `technology_type`-related columns appear to track each plant generator's fuel consumption.  
    - In this context, what does the value `all` in `fuel_consumption_from_all_fuel_types` represent?  
    - If `fuel_consumption_from_all_fuel_types` is **different** from the single tech-specific value, then the grain for these dynamic metrics may not be an individual plant generator.

#### **Next Steps**
- Compare `fuel_consumption_from_all_fuel_types` to the corresponding technology-specific fuel consumption to clarify the grain. 
- Investigate `cooling_id` and `cooling_unit_hours_in_service` to determine whether the cooling unit defines a finer-grained entity in the dataset.

> ```
> cooling_df[['fuel_consumption_from_all_fuel_types_mmbtu',
>             'fuel_consumption_from_steam_turbines_mmbtu',
>             'fuel_consumption_from_single_shaft_combined_cycle_units_mmbtu',
>             'fuel_consumption_from_combined_cycle_gas_turbines_mmbtu']].drop_duplicates().head(50)
> ```

#### **Notes**
- `_all_fuel_types` appears redundant; most likely a pre-coalesced column for query convenience.  
- The technology type columns narrate each unique plant generator.  
- Since this dataset is about **thermoelectric cooling** rather than generation, we hypothesize that the central entity relates to **water used for regulating fuel-consuming technologies**, rather than fuel consumption itself.

#### **Questions**
- If water metrics (consumption, withdrawal, intensity, etc.) track dynamic metrics of the cooling system, what defines a cooling system?  
    - Columns `cooling_type_1_860`, `cooling_type_2_860`, `cooling_type_923`, `cooling_system_type` appear to be dimension attributes of the cooling system.  
    - There is **no explicit `cooling_system_id`**.  
    - Should each unique `plant_id`-`cooling_id` pair have only one cooling system type, similar to how each unique `plant_id`-`generator_id` pair has only one technology type?  
- What is the relationship between `generator_id`, `boiler_id`, and `cooling_id`? How many unique units exist for each?  
    - Investigate `relationship_type`; this column strongly implies that the combination of these three IDs defines the row-level grain.  
    - Does each unique `plant_id`-`generator_id`-`boiler_id`-`cooling_id` combination have one or more `relationship_type` values?

> ```
> Cooling system types per (plant_code, cooling_id):
> cooling_system_type
> 0     110
> 1    1950
> 2      46
> 3       1
> Name: count, dtype: int64
> ```
- 47 plant-cooling pairs have multiple cooling_system_type values — possibly time-sequenced (SCD Type 2) or a cooling unit part of more than one `generator_id` and `boiler_id` configuration? (Unsure if this is possible simultaneously)

> ```
> Relationship types per (plant, gen, boiler, cooling):
> relationship_type
> 1    5965
> 2    1449
> 3      55
> 4       4
> Name: count, dtype: int64
> ```
- 1,508 equipment configurations have multiple relationship_type values -- this might also be SCD, not grain issue.

###   ***2b) Conclusion: Grain candidate `plant_code`-`generator_id`-`boiler_id`-`cooling_id`-`year`-`month`*** 
> ```
> Grain hunt:
> Total rows: 917436
> (plant, gen, boiler, cooling): 7473
> (plant, gen, boiler, cooling, year, month):917016
> (plant, gen, boiler, cooling, relationship_type, year, month):917016
> ``` 
- Adding relationship_type to the grain doesn't increase uniqueness; this confirms that it's time-sequenced (one value per month). It's an SCD attribute, not part of the composite key.  
- 99.95% match rate between total rows of dataset (917,436) and composite unique combos (917,016). 
- Next step: investigate the possibly duplicate 420 rows. 

> ```
> null_grain[metric_cols].describe()
> ```

#### Notes

- `water_withdrawal_volume_million_gallons`: Values span a wide range with a non-zero minimum. The mean is close to the median, and the maximum is not dramatically larger than the upper quartile, suggesting moderate spread with no strong skew (possibly mild right-skew), but broadly symmetric overall.

- `water_consumption_volume_million_gallons`: Values range from 0 to 1, with the majority of observations (up to the 75th percentile) equal to 0. This raises the question of whether near-zero consumption can occur alongside non-zero water withdrawal in thermoelectric operations. Since water withdrawal reflects total water drawn from a source while consumption reflects water not returned (e.g., due to evaporation), this pattern may be explained by cooling system design or operational status. This will be explored further alongside `relationship_type` and `cooling_system_type`.

- `fuel_consumption_from_all_fuel_types`: Values range from 0 to ~ 300K mmbtu. The mean (~ 109K) is substantially higher than the median (~ 33K), indicating right-skew driven by a minority of large values. This is further supported by the 75th percentile (~ 217K), which is much closer to the upper tail than to the median.

#### TD;LR: 
> Based on these summary statistics, these rows appear to contain meaningful water usage information and should not be dismissed solely due to missing identifiers.

---
# 3. **Data Profiling & Investigating**  
##   ***3a) Investigating the 420-Row Discrepancy***                                                          
> ```
> Duplicate grain combinations: 0
> Total extra rows: 0
>
> Duplication counts:
> Series([], Name: count, dtype: int64)
> ```
- `groupby()` silently drops rows with any NULLs in any grouping column. We'll look at whether the missing 420 rows are duplicates, but on account of NULL values in one or more of the composite grain columns.

> ```
> NULLs in grain columns:
> plant_code        0
> generator_id    396
> boiler_id       156
> cooling_id       24
> year              0
> month             0
> dtype: int64
>
> Rows with NULL in at least one grain column: 420
> ```
- The 420 missing rows are due to NULL values in the grouping columns.
➡️ Discovery: Not duplicates — rows with NULL grain values excluded from groupby()

#### Next Steps: 
To decide whether and how to clean them, we will investigate:

1. What’s going on with the water metrics and overall fuel consumption for these rows?  
2. Is there a pattern in `*_status` or `relationship_type` for these rows?  
3. ~Would deleting them bias representation of a state, plant, or `technology_type`?~ (*deprecated after exploring questions 1 & 2*)

##   ***3b) Examine water metrics & overall fuel consumption***
> ```
> === Q1: Metrics coverage in NULL-grain rows ===
> water_withdrawal_volume_million_gallons: 216 rows (51.4%)
> water_consumption_volume_million_gallons: 216 rows (51.4%)
> fuel_consumption_from_all_fuel_types_mmbtu: 24 rows (5.7%)
> 
> === Q1: Metrics coverage in full dataset ===
> water_withdrawal_volume_million_gallons: 606172 rows (66.1%)
> water_consumption_volume_million_gallons: 567951 rows (61.9%)
> fuel_consumption_from_all_fuel_types_mmbtu: 724004 rows (78.9%)
> ```
- Hypothesis: These rows may reflect **water use independent of active fuel consumption or energy production**.This might be an anomaly to flag or a feature of electricity generation. (Domain research required.)  
- Next step: Summarize the actual metric values for these rows to see ranges, medians, or unusual patterns.

##   ***3c) Examine equipment status & relationship type***  
#### Notes
- For the subset of rows with missing equipment identifiers, the `relationship_type` is consistently **'Unoperable'**.
    - However, there are ~175K rows with an 'Unoperable' `relationship_type`, including rows with a complete composite grain.
- The `*_status` columns for the Unoperable–incomplete grain subset contain values beyond `NaN`, including `'OP'`, which we currently infer to mean *Operable*.
    - Regardless of exact code meanings, this variety suggests a nuanced system of equipment description that can be read across to tell a detailed story for each configuration.
    - In other words, much as the distribution of water and fuel consumption metrics has suggested, these rows contain meaningful information despite missing identifiers.

#### Next Steps
1) Perform a quick lookup of human-interpretable descriptions for the EIA status codes (remapping not immediately required).
2) Normalize the `*_status` columns for:
    - the Unoperable–incomplete grain subset (which is the same as the duplicates subset) vs.
    - the Unoperable–complete grain subset
3) Compare value rates for each `*_status` column across subsets  
  *(count of value ÷ total rows in subset)*.
   
###        *3c1) EIA Status Codes Lookup*    
Gemini AI - 01/22/2026: 

"Those status codes appear in the **EIA-860** and **EIA-923** surveys, which track the lifecycle and operational state of power plant equipment (boilers, generators, and cooling systems).

#### Operational & Retired Status

* **OP**: **Operating** – In commercial service.
* **SB**: **Stand-by** – Long-term storage; available for service but not normally used.
* **SC**: **Cold Stand-by** – Deactivated; reserve status (often requires more time to return to service than SB).
* **OA**: **Out of Service** – Temporarily out of service, but expected to return to service (usually for maintenance).
* **OS**: **Out of Service** – Not expected to be returned to service (often a precursor to retirement).
* **RE**: **Retired** – No longer in service and not expected to return.

#### Construction & Development Status

* **CO**: **New unit under construction**.
* **U**: **Under construction** – Less than 50% completed.
* **V**: **Under construction** – More than 50% completed.
* **TS**: **Testing** – Construction is complete, but the unit is not yet in commercial operation (includes low-power testing).

#### Planning & Regulatory Status

* **PL**: **Planned** – Expected to go into commercial service within 10 years.
* **P**: **Planned** – Regulatory approvals have not yet been initiated.
* **L**: **Regulatory approvals pending** – Not yet under construction.
* **T**: **Regulatory approvals received** – Not yet under construction.
* **CN**: **Cancelled** – Previously reported as "planned" but now cancelled.
* **IP**: **Indefinitely Postponed** – Planned new generator is postponed or no longer in the resource plan."

###        *3c2) Normalization & Rate Comparison: `*_status` Columns*  

#### Unoperable-Complete Grain vs. Unoperable-Incomplete Grain Subsets

- In the Unoperable–Incomplete Grain subset, `generator_status` is `NaN` at a dramatically higher rate (near 100%) than in the Unoperable–Complete Grain subset.  
  - *Note:* This does **not** necessarily imply a missing `generator_id`; “Unoperable–Incomplete–Missing `generator_id`” would define a distinct subset.
- A similarly large difference is observed for `boiler_status`.

- By contrast, the rate of `NaN` values for `cooling_status` is effectively the same between the Unoperable–Incomplete and Unoperable–Complete subsets.
- Moreover, `cooling_status` in the Unoperable–Incomplete subset is **Operable** at a substantially higher rate than in the Unoperable–Complete subset (54.3% vs. 17.5%).

- For all three `*_status` columns, the most frequent value in the Unoperable–Complete subset is `'RE'` (**Retired**), a value that is effectively absent in the Unoperable–Incomplete subset.

- Taken together, these results suggest an asymmetry across equipment types:  
  - In the Unoperable–Incomplete subset, `boiler_status` and `generator_status` are predominantly missing,  
  - while `cooling_status` is frequently **Operable**.
- This pattern may help explain why rows with missing equipment identifiers still produce meaningful thermoelectric measurement data.

- Notably, this configuration—meaningful measurements from cooling systems that are marked Unoperable yet lack equipment identifiers—appears most frequently among **Closed** cooling systems.
  - *Note:* Closed systems are the dominant cooling system type in the dataset; additional normalization and statistical testing will be required to determine whether this pattern represents a significant difference by cooling system type.
- With respect to the representation and significance of `cooling_system_type`, chi-squared testing will be deferred to the formal EDA phase.

##   ***3d) Conclusion: Grain confirmed, incomplete rows are systematic (not noise)***
- Duplicate rows arise from `NULL` values in composite grain identifiers.
- These rows should **not** be removed solely due to missing equipment identifiers, nor because `relationship_type` is consistently Unoperable:
  - Water and fuel consumption metrics continue to exhibit meaningful values.
  - Approximately 50% of the time, the cooling unit remains **Operable**.
- Duplicate rows are **not** attributable to `NULL` values in other grain attributes.
- To complete data cleaning, we will examine remaining grain-related columns for potential aberrations:
  - `plant_code`
  - `state`
  - `plant_name`
- While `plant_code` contains no missing values, both `state` and `plant_name` contain 24 missing rows and warrant further inspection.

---

# 4. **Plant-Level Validation & Paradox Resolution**  
##   ***4a) Plant-Level Validation***  
> ```
> # Boolean overlap check
> missing_state = cooling_df['state'].isna()
> missing_plant = cooling_df['plant_name'].isna()
> 
> # How many rows have both missing?
> (missing_state & missing_plant).sum()
> ```
- Rows with NULL `state` are identical to those with NULL `plant_name`
 
#### Story of the 24 rows with missing `state` and `plant_name`

- The same `utility_id` (58371) and `plant_code` (57273)
- The rows comprise of 2 different observation sets across the `year` 2015.
- They are observations for two different grain-composites, differentiated by different `boiler_id` values (A and B); `generator_id` is NULL for both grains; the `cooling_id` is '1' for both cases.
- All `technology_type`-related columns are NULL (`technology_type` is therefore 'None')
    - A quick check reveals the number of 'None' values for `technology_type` (3,930) vastly exceeds the number of rows missing `generator_id` (396).
- All water metrics and cooling system meta-data are also NULL.
- Both boilers have `boiler_status` 'CN', meaning cancelled, though boiler 'B' has a `boiler_inservice_month` of '12' (December) and `boiler_inservice_year` of '2016'. **This seems paradoxical**, given that the year for these observations is '2015'
    - Are there other rows where the `*_inservice_` date is later than the row's observation date?
    - How does EIA manage the timeliness of these datasets?
- `cooling_status` is 'CN' across.
- The relevant `sector` for the plant is 'IPP Non-CHP sector': Independent Power Producer in the Non-Combined Heat Power Sector (private electricity generation entity that does not use waste heat for industrial or heating purposes); the `steam_plant_type` information is NULL, which means that the rows with missing `state` and `plant_name` are also the same 24 missing `steam_plant_type`
- The `relationship_type` is 'Unoperable'.

#### Next Steps: 

- See if the `utility_id`-`plant_code` pair have an associated `plant_name` and `state` elsewhere in the dataset.
    - If not, try `plant_code` alone.
- Consider whether rows with this much missing metric information remain useful for any kind of analysis.
    - How many rows in the data are similarly sparse and/or paradoxical--with `*_inservice_` date information, particularly after the observation date, while 'Unoperable' with zero measurement metrics? 

#### 1. Can we infer `plant_name` and/or `state` based on `plant_code` and `utility_id`
#### 2. Explore 'fingerprint' for 'Blythe Solar Power Project' in California
- The observed dates include 2014. They may correspond to the same plant as the 24 rows with missing state and plant_name in 2015.  
- How many unique ID's exist? How do their `*_status` values distribute?
  
> ```
> --- generator_id ---
>   generator_id generator_status  row_count
> 0            1               IP         24
> 1            2               IP         24
> 2            3               IP         24
> 3            4               IP         24
> 
> --- boiler_id ---
>   boiler_id boiler_status  row_count
> 0         A            CN         48
> 1         B            CN         48
> 
> --- cooling_id ---
>   cooling_id cooling_status  row_count
> 0          1             CN         96
> ```

- The four generators are 'IP' (Indefinitely Postponed) while the other components are 'CN' (Cancelled - previously reported as Planned)

#### Notes: Blythe

- The 'Indefinitely Postponed' generator was slated to use steam turbine technology (`summer_capacity_of_steam_turbines_mw` = 232).
- `steam_plant_type` is '4.0' (see decision_appendix.ipynb for code explanation): Solar Thermal or Other.
- The `boiler_inservice` date shows the same paradox as the 24 mystery rows: December 2016, later than the observation dates.
- The `sector` is 'IPP Non-CHP', just like for the 24 rows. 
- Observation: The 24 rows with missing plant_name and state share the same utility_id and plant_code, and their characteristics closely match Blythe Solar Power Project (CA).

#### Next Steps
- Impute plant_name and state
- Decide whether to include plants like Blythe (with postponed or non-operational equipment) in the analysis.

###        *4a1) Imputation*  
##   ***4b) Explore "Paradox Timelines" across the dataset***
_Blythe puts our inquiry at a plant level_
- Check how many rows have in-service dates later than observation dates.  
- This could affect our analysis strategy depending on how common this pattern is.
#### 1. Create `cooling_subset`: subset of rows with at least one `*_inservice_` month/year
- Only 1,104 rows have no equipment inservice date information at all. 
#### 2. Enrich `cooling_subset`: convert `year`/`month` and available `*_inservice_` year/month to datetime (+4 columns)
#### 3. Create `master_paradox`: subset of `cooling_subset` containing only rows with timeline paradox
(i.e. `*_inservice_` date > `observation_date`)
- With 15,017 rows,'paradox timeline' rows represent ~ 1.64% of the dataset; they involve data for 117 unique plants (~ 11.46% of covered U.S. power plants).
#### 4. Explore `master_paradox`: Notes on Paradox Timeline Rows 

**Categorical Column Summary:** 

- 117 unique `plant_code` values (118 `plant_name`, implying inconsistencies) across 35 unique `state` values.
    - `plant_code` is the join/grouping key, while `plant_name` will be display-only unless normalized.
- `generator_primary_technology` is overwhelmingly 'Natural Gas Fired Combined Cycle' (~92% of rows).
- Slightly over 50% of `cooling_system_type` values are 'Closed' (recirculating with induced draft).
- 100% of rows have a `relationship_type` of 'Unoperable'.
- Overview of `*_status` columns shows diversity: no single value dominates, with 10 different `generator_status`, 7 different `boiler_status`, and 6 different `cooling_status`.

**Numerical Column Summary / Descriptive Statistics:** 

- Observations span the full dataset timeline (2014–2024).
- `technology_type`-related dimension columns are extremely sparse but contain meaningful information. Note negative min values (~–4K MWh) for both `net_generation_from_steam_turbines_mwh` and `net_generation_associated_with_combined_cycle_gas_turbines_mwh` — may be anomalous; requires domain investigation.
- Primary water metrics (`water_withdrawal_volume_million_gallons` and `water_consumption_volume_million_gallons`) are also extremely sparse (>90% missing) but valid metric values where present.
- Equipment `*_inservice_` and `*_retirement_` dates:
    - **Generators:** Max in-service date ≤ 2024 cut-off; no retired generators in this subset.
    - **Boilers:** Max in-service date **exceeds** dataset cut-off (2027); retirement max = 2066.
    - **Cooling units:** Max in-service date also **exceeds** dataset cut-off (2028); no retirement dates present.

#### TL;DR: 
Timeline paradox rows form a small but internally consistent subset of the dataset. They are characterized by homogeneous technology profiles, uniform `relationship_type` values, and temporally forward-dated equipment metadata relative to observation periods.

These rows do not appear to be random data quality errors, but their temporal misalignment makes them unsuitable for naïve time-series aggregation. They will be **retained but flagged.** 

---

# 5. **Melt dataset: `cooling_df_2`**  
##   ***5a) Generating Paradox Timeline Flags***  
##   ***5b) Water Metric Derivations***  
- By restructuring the dataset, we fill in missing fuel consumption and power generation values across additional rows. From these, we can compute the sparse_intensity and rate_per_fuel metrics.

---

# 6. **Conclusion**
                                                                                              
  This profile analysis characterized the structure and quality of the EIA
thermoelectric cooling dataset (2014–2024) and prepared it for downstream
water use analysis. **Key takeaways:**

- **Grain confirmed:** The observation grain is `(plant_code, generator_id,
  boiler_id, cooling_id, year, month)` with **99.95% consistency** across 917,436
  rows.
- **Incomplete rows are systematic, not noise:** ~ 420 rows with NULL grain
  values and ~ 15,017 rows (**~ 1.64%**) with paradox timelines both trace back to
  "Unoperable" equipment relationships — they are expected and interpretable
  patterns.
- **Missing metadata is recoverable:** 24 rows with missing
  `state`/`plant_name` can be resolved via `utility_id` and `plant_code` (all
  map to the **Blythe Solar Power Project, CA**).
- **Temporal paradoxes are flag-able and bounded:** Paradox rows affect **117
  plants (~ 11.5% of covered plants)** and are almost entirely gas turbine
  combined-cycle facilities; paradox flags have been added to the dataset.
- **Water metric coverage improved dramatically:** Derived calculations
  expanded usable water intensity coverage from **5.3% → 54.0%** (**~ 924% gain**),
  enabling analysis across ~ 496K rows instead of ~48K.

The exported dataset (`cooling_melted_df.csv`) is cleaned, enriched with
paradox flags and derived water metrics, and ready for analysis in subsequent
notebooks.
